In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce



def handle_schema_drift(df):

    df = df.withColumn(
        "rescued_map",
        from_json(col("_rescued_data"), MapType(StringType(), StringType()))
    )

    rescued_keys = (
        df.selectExpr("explode(map_keys(rescued_map)) as key")
          .distinct()
          .rdd.flatMap(lambda x: x)
          .collect()
    )

    ignore_keys = ["_file_path", "_corrupt_record"]
    schema_dict = {field.name.lower(): field.dataType for field in df.schema.fields}

    for key in rescued_keys:
        if key.lower() in ignore_keys:
            continue

        normalized_key = key.lower()
        matched_col = None

        if normalized_key in schema_dict:
            matched_col = normalized_key
        else:
            for existing_col in df.columns:
                if existing_col.lower() in normalized_key or normalized_key in existing_col.lower():
                    matched_col = existing_col
                    break

        if matched_col:
            target_type = schema_dict.get(matched_col.lower())
            df = df.withColumn(
                matched_col,
                coalesce(col(matched_col), col("rescued_map")[key].cast(target_type))
            )
        else:
            df = df.withColumn(normalized_key, col("rescued_map")[key])

    return df.drop("rescued_map", "_rescued_data")



def apply_data_quality(df, pk_col, trim_cols):

    for c in trim_cols:
        df = df.withColumn(c, trim(col(c)))

    df = df.select([col(c).alias(c.lower()) for c in df.columns])

    reject_df = df.filter(col(pk_col).isNull())
    df = df.filter(col(pk_col).isNotNull())

    df = df.fillna("unknow")

    window = Window.partitionBy(pk_col).orderBy(current_timestamp().desc())

    df = df.withColumn("rn", row_number().over(window)) \
           .filter(col("rn") == 1) \
           .drop("rn")

    return df, reject_df



def apply_scd2(df, table_name, pk, sk_col):

    from pyspark.sql.window import Window
    from delta.tables import DeltaTable

    # Normalize pk to a list
    if isinstance(pk, str):
        pk = [pk]

    df = df.withColumn("StartDate", current_timestamp()) \
           .withColumn("EndDate", lit(None).cast("timestamp")) \
           .withColumn("IsCurrent", lit(1)) \
        #    .withColumn("ingestion_time", current_timestamp())

    compare_cols = [c for c in df.columns if c not in ["StartDate","EndDate","IsCurrent"]]

    df = df.withColumn(
        "hash",
        sha2(concat_ws("||", *compare_cols), 256)
    )

    if not spark.catalog.tableExists(table_name):

        window = Window.orderBy(pk)

        df = df.withColumn(
            sk_col,
            row_number().over(window)
        )

        df.write.format("delta").saveAsTable(table_name)

    else:
        delta_table = DeltaTable.forName(spark, table_name)

        target_df = delta_table.toDF()

        # Add sk_col to source so whenNotMatchedInsertAll can resolve all columns
        max_id = target_df.agg({sk_col: "max"}).collect()[0][0]
        max_id = max_id if max_id else 0
        window_sk = Window.orderBy(pk)
        df = df.withColumn(sk_col, row_number().over(window_sk) + max_id)

        merge_cond = " AND ".join([f"t.`{p}` = s.`{p}`" for p in pk]) + " AND t.IsCurrent = 1"

        delta_table.alias("t").merge(
            df.alias("s"),
            merge_cond
        ).whenMatchedUpdate(
            condition="t.hash <> s.hash",
            set={
                "EndDate": current_timestamp(),
                "IsCurrent": lit(0)
            }
        ).whenNotMatchedInsertAll().execute()

        # Re-read target after merge to get updated max_id
        updated_target = spark.table(table_name)

        join_cond = [col(f"s.`{p}`") == col(f"t.`{p}`") for p in pk]
        join_cond.append(col("t.IsCurrent") == 0)
        join_expr = reduce(lambda a, b: a & b, join_cond)

        changed_df = df.alias("s").join(
            target_df.alias("t"),
            join_expr,
            "inner"
        ).select("s.*").dropDuplicates(pk + ["hash"])

        new_max_id = updated_target.agg({sk_col: "max"}).collect()[0][0]
        new_max_id = new_max_id if new_max_id else 0

        window = Window.orderBy(pk)

        changed_df = changed_df.drop(sk_col).withColumn(
            sk_col,
            row_number().over(window) + new_max_id
        )

        changed_df.write.format("delta").mode("append").saveAsTable(table_name)



# bronze_titles = spark.table("titles")
bronze_cast = spark.table("`adb-netflix`.`bronze-layer`.`cast`")
bronze_directors = spark.table("`adb-netflix`.`bronze-layer`.`directors`")







cast_df = handle_schema_drift(bronze_cast)

cast_clean, _ = apply_data_quality(
    cast_df, "show_id", ["cast"]
)

apply_scd2(
    df=cast_clean,
    table_name="`adb-netflix`.`silver-layer`.cast",
    pk=["cast"],
    sk_col="cast_id"
)




director_df = handle_schema_drift(bronze_directors)

director_clean, _ = apply_data_quality(
    director_df, "show_id", ["director"]
)

apply_scd2(
    df=director_clean,
    table_name="`adb-netflix`.`silver-layer`.directors",
    pk=["director"],
    sk_col="director_id")


print("Pipeline Completed with SCD Type 2")

In [0]:
# bronze_titles = spark.table("`adb-netflix`.`bronze-layer`.`titles`")

# titles_df = handle_schema_drift(bronze_titles)

# titles_clean, titles_reject = apply_data_quality(
#     titles_df, "show_id", ["title","type"]
# )

# silver_titles = titles_clean \
#     .withColumn("release_year", col("release_year").cast("int")) \
#     .withColumn("ModifiedDate", current_timestamp())

# silver_titles.write.mode("overwrite").saveAsTable("silver_titles")

In [0]:
bronze_countries = spark.table("`adb-netflix`.`bronze-layer`.`countries`")
bronze_category = spark.table("`adb-netflix`.`bronze-layer`.`category`")

countries_df = handle_schema_drift(bronze_countries)
silver_countries, _ = apply_data_quality(
    countries_df, "show_id", ["country"]
)




silver_countries.write.mode("overwrite").saveAsTable("`adb-netflix`.`silver-layer`.countries")

category_df = handle_schema_drift(bronze_category)

silver_category, _ = apply_data_quality(
    category_df, "show_id", ["listed_in"]
)

silver_category.write.mode("overwrite").saveAsTable("`adb-netflix`.`silver-layer`.category")